# Lab 3 - Named Entity Recognition for voice-assistant transcripts

**Topic:** NER (PERSON, LOCATION, ORGANIZATION, DATE) on speech-to-text transcripts.
A from-scratch linear-chain CRF with Viterbi decoding is compared with a BiLSTM + char-CNN tagger (PyTorch).

Source code (imported, not modified): `lab 3/solution/ner_crf_lstm.py` and `lab 3/solution/ner_data.py`.

| Task | Topic | Section in this notebook |
|---|---|---|
| 1 | Convert the transcripts into BIO tags | TASK 1 |
| 2 | Word-level features (CRF) and embeddings (LSTM) | TASK 2 |
| 3 | CRF-based NER model (written from scratch) | TASK 3 |
| 4 | Viterbi decoding (from scratch, checked by brute force) | TASK 4 |
| 5 | LSTM-based NER model (BiLSTM + char-CNN) | TASK 5 |
| 6 | Test both models on unseen speech transcripts | TASK 6 |
| 7 | Compare predictions and evaluation metrics | TASK 7 |
| 8 | Analyse a case where the models disagree | TASK 8 |
| 9 | Evaluate with appropriate metrics (entity-level P/R/F1, seqeval, lower-cased ASR test) | TASK 9 |

Kernel: Speech Lab (.venv)

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from sklearn.feature_extraction import DictVectorizer

# The notebook lives in notebooks/; the repo root is found by searching upwards from the
# working directory (robust to re-running this cell after os.chdir below).
_here = Path.cwd()
ROOT = next(p for p in [_here, *_here.parents] if (p / "lab 3" / "solution").exists())
SOL = ROOT / "lab 3" / "solution"
assert SOL.exists(), SOL
sys.path.insert(0, str(SOL))
os.chdir(SOL)

from ner_data import (TAGS, ENTITY_TYPES, TRAIN_TRANSCRIPTS, TEST_TRANSCRIPTS,
                      load_dataset, strip_markup, tokenize)
from ner_crf_lstm import (TAG2IDX, IDX2TAG, banner, get_entities, evaluate, print_report,
                          sent2features, word2features, Vocab, train_crf, viterbi_decode,
                          brute_force_best, crf_predict, train_lstm, lstm_predict,
                          entities_str, print_aligned, transcribe_audio)

## TASK 1 - Convert the transcripts into BIO tags

Each inline-annotated transcript such as `Dr [Arun Kumar|PER]` is tokenised and every token gets a tag:
`B-X` for the first token of an entity, `I-X` for the following tokens, and `O` otherwise.
The 60 training and 20 unseen test transcripts are parsed with `load_dataset`.

In [2]:
banner("TASK 1 - Convert the transcripts into BIO tags")
train = load_dataset(TRAIN_TRANSCRIPTS)
test = load_dataset(TEST_TRANSCRIPTS)
print(f"Training transcripts: {len(train)} | unseen test transcripts: {len(test)}")
print(f"Tag set ({len(TAGS)}): {TAGS}\n")
for toks, tags in train[:3]:
    print("  " + " | ".join(f"{w}/{t}" for w, t in zip(toks, tags)))
tag_counts = Counter(t for _, tags in train for t in tags)
print("\nTag distribution in training data:", dict(sorted(tag_counts.items())))


TASK 1 - Convert the transcripts into BIO tags
Training transcripts: 60 | unseen test transcripts: 20
Tag set (9): ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-DATE', 'I-DATE']

  Book/O | a/O | flight/O | for/O | Dr/O | Arun/B-PER | Kumar/I-PER | from/O | Chennai/B-LOC | to/O | Singapore/B-LOC | on/O | 15/B-DATE | September/I-DATE
  Schedule/O | a/O | meeting/O | with/O | Priya/B-PER | Sharma/I-PER | at/O | Infosys/B-ORG | on/O | Monday/B-DATE
  Call/O | Rahul/B-PER | and/O | tell/O | him/O | I/O | will/O | reach/O | Mumbai/B-LOC | tomorrow/B-DATE

Tag distribution in training data: {'B-DATE': 43, 'B-LOC': 46, 'B-ORG': 33, 'B-PER': 30, 'I-DATE': 27, 'I-LOC': 5, 'I-ORG': 19, 'I-PER': 20, 'O': 366}


## TASK 2 - Word-level features (CRF) and embeddings (LSTM)

The CRF receives sparse hand-crafted features per token (shape, capitalisation, suffixes, month/weekday/title
lookups, and a +-2 word context window), vectorised with `DictVectorizer`. The LSTM instead learns a 50-d word
embedding plus a character-CNN embedding, so it can build a vector for words never seen in training.

In [3]:
banner("TASK 2 - Word-level features (CRF) and embeddings (LSTM)")
vec = DictVectorizer(sparse=False)
train_feats = [sent2features(t) for t, _ in train]
vec.fit([f for s in train_feats for f in s])
to_tensor = lambda feats: torch.tensor(vec.transform(feats), dtype=torch.float32)
X_train = [to_tensor(f) for f in train_feats]
y_train = [torch.tensor([TAG2IDX[t] for t in tags]) for _, tags in train]
print(f"CRF feature space: {len(vec.feature_names_)} sparse binary/real features")
example = word2features(train[0][0], 5)
print(f"Feature dictionary for the token '{train[0][0][5]}' in '{' '.join(train[0][0])}':")
for k in list(example)[:16]:
    print(f"    {k:<22} = {example[k]}")
print("    ...")

vocab = Vocab([t for t, _ in train])
print(f"\nLSTM vocabulary: {len(vocab.w2i)} words (+<PAD>,<UNK>), {len(vocab.c2i)} characters")
print("LSTM token representation = word embedding (50-d, learned) + char-CNN embedding (30-d)")
print("  -> the char-CNN lets the model build a vector for words never seen in training")


TASK 2 - Word-level features (CRF) and embeddings (LSTM)
CRF feature space: 2022 sparse binary/real features
Feature dictionary for the token 'Arun' in 'Book a flight for Dr Arun Kumar from Chennai to Singapore on 15 September':
    bias                   = 1.0
    w.lower                = arun
    w.prefix3              = aru
    w.suffix2              = un
    w.suffix3              = run
    w.shape                = Xxx
    w.istitle              = True
    w.isupper              = False
    w.isdigit              = False
    w.is_month             = False
    w.is_weekday           = False
    w.is_relative_day      = False
    w.is_org_suffix        = False
    w.is_ordinal           = False
    -2:w.lower             = for
    -2:w.istitle           = False
    ...

LSTM vocabulary: 298 words (+<PAD>,<UNK>), 61 characters
LSTM token representation = word embedding (50-d, learned) + char-CNN embedding (30-d)
  -> the char-CNN lets the model build a vector for words never seen in 

## TASK 3 - CRF-based NER model (linear-chain CRF, from scratch)

The CRF is a PyTorch module with emission weights (linear in the features), a tag-transition matrix,
and start/end scores. The log-partition function is computed with the forward algorithm, the negative
log-likelihood is minimised with L-BFGS plus L2 regularisation, and gradients come from autograd.
The cell prints the learned transition matrix and the most strongly weighted features per tag.

In [4]:
banner("TASK 3 - CRF-based NER model (linear-chain CRF, from scratch)")
crf = train_crf(X_train, y_train, len(vec.feature_names_))
trans, start, end = crf.numpy_params()
print("\nLearned transition scores (row = previous tag, column = next tag):")
print("         " + "".join(f"{t:>8}" for t in TAGS))
for i, t in enumerate(TAGS):
    print(f"{t:>8} " + "".join(f"{trans[i, j]:8.2f}" for j in range(len(TAGS))))
W = crf.emit.weight.detach().numpy()
names = vec.feature_names_
print("\nTop-5 features per tag (highest emission weights):")
for k, t in enumerate(TAGS):
    top = np.argsort(-W[k])[:5]
    print(f"  {t:<7}: " + ", ".join(f"{names[i]} ({W[k, i]:.2f})" for i in top))


TASK 3 - CRF-based NER model (linear-chain CRF, from scratch)


CRF trained with L-BFGS (L2 c2=0.05) in 2.3s, final objective = 1293.722

Learned transition scores (row = previous tag, column = next tag):
                O   B-PER   I-PER   B-LOC   I-LOC   B-ORG   I-ORG  B-DATE  I-DATE
       O     0.56    1.89   -1.10    1.23   -0.69    1.05   -1.65    0.58   -1.84
   B-PER    -0.50   -0.66    2.81   -0.27   -0.48   -0.28   -0.70    0.35   -0.28
   I-PER     0.55   -0.04   -0.08   -0.02   -0.01   -0.04   -0.08   -0.23   -0.05
   B-LOC    -0.05   -0.54   -0.52   -0.11    1.85   -0.12   -0.78    0.60   -0.36
   I-LOC     0.16   -0.03   -0.05   -0.05   -0.01   -0.01   -0.06    0.14   -0.10
   B-ORG     0.40   -0.42   -0.62   -0.15   -0.50   -0.29    2.20   -0.39   -0.26
   I-ORG    -0.29   -0.10   -0.22   -0.25   -0.10   -0.18    1.31    0.11   -0.26
  B-DATE    -0.53   -0.12   -0.22   -0.37   -0.08   -0.12   -0.21   -0.69    2.34
  I-DATE    -0.27   -0.01   -0.02   -0.03   -0.00   -0.01   -0.03   -0.46    0.85

Top-5 features per tag (highest emissi

## TASK 4 - Viterbi decoding (best tag sequence)

Viterbi is implemented in NumPy: `delta[t, j]` holds the best score of any path ending in tag `j` at position `t`,
with back-pointers `psi`. The trellis of the first unseen test sentence is printed, and on the shortest test
sentence the Viterbi result is checked against brute-force enumeration of all K^T tag paths. The
sklearn-crfsuite reference CRF is also trained here (as in the script) and is used for comparison in TASK 7.

In [5]:
banner("TASK 4 - Viterbi decoding (best tag sequence)")
demo_toks, demo_gold = test[0]
with torch.no_grad():
    em = crf.emissions(to_tensor(sent2features(demo_toks))).numpy()
path, best_score, delta, psi = viterbi_decode(em, trans, start, end)
print("Sentence:", " ".join(demo_toks))
print("\nViterbi trellis delta[t, tag] (best score of any path ending in tag at position t):")
print(f"  {'token':<12}" + "".join(f"{t:>8}" for t in TAGS) + "   best")
for t, tok in enumerate(demo_toks):
    print(f"  {tok:<12}" + "".join(f"{delta[t, j]:8.1f}" for j in range(len(TAGS)))
          + f"   {IDX2TAG[int(delta[t].argmax())]}")
print("\nBack-tracked best path:", [IDX2TAG[i] for i in path])
print(f"Best path score       : {best_score:.3f}")

# verify Viterbi against exhaustive search on the shortest test sentence
short_toks = min((t for t, _ in test), key=len)
with torch.no_grad():
    em_s = crf.emissions(to_tensor(sent2features(short_toks))).numpy()
if len(short_toks) <= 6:
    vp, vs, _, _ = viterbi_decode(em_s, trans, start, end)
    bp, bs = brute_force_best(em_s, trans, start, end)
    print(f"\nCorrectness check on '{' '.join(short_toks)}' "
          f"({len(TAGS)}^{len(short_toks)} = {len(TAGS) ** len(short_toks):,} paths):")
    print(f"  Viterbi    : {[IDX2TAG[i] for i in vp]}  score={vs:.4f}")
    print(f"  Brute force: {[IDX2TAG[i] for i in bp]}  score={bs:.4f}")
    print("  -> identical:", vp == bp, "| Viterbi cost O(T*K^2) instead of O(K^T)")

# optional: compare against the sklearn-crfsuite library CRF (if installed)
crfsuite = None
try:
    import sklearn_crfsuite
    crfsuite = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05,
                                    max_iterations=200, all_possible_transitions=True)
    crfsuite.fit(train_feats, [tags for _, tags in train])
    print("\n(sklearn-crfsuite reference CRF also trained for comparison)")
except ImportError:
    print("\n(sklearn-crfsuite not installed - skipping library CRF reference)")


TASK 4 - Viterbi decoding (best tag sequence)
Sentence: Book a flight for Dr Meera Krishnan from Ahmedabad to Toronto on 12 August

Viterbi trellis delta[t, tag] (best score of any path ending in tag at position t):
  token              O   B-PER   I-PER   B-LOC   I-LOC   B-ORG   I-ORG  B-DATE  I-DATE   best
  Book            12.0    -0.9    -2.1    -1.5    -1.7    -1.3    -0.8    -2.3    -1.4   O
  a               19.6    13.1     9.8    10.5    10.3    12.4    11.2    12.3     9.3   O
  flight          27.8    18.9    17.6    20.3    17.1    19.4    18.1    19.6    17.9   O
  for             37.5    28.1    25.1    26.9    25.0    27.2    26.5    28.3    25.9   O
  Dr              42.7    38.4    35.2    39.1    35.3    38.3    36.1    38.1    34.4   O
  Meera           43.5    47.1    41.7    42.2    42.1    43.5    42.5    41.3    40.3   B-PER
  Krishnan        45.8    46.4    53.3    47.8    46.0    46.5    47.4    45.5    45.5   I-PER
  from            61.9    51.7    51.8    50


Correctness check on 'How far is Mysore from Bangalore' (9^6 = 531,441 paths):
  Viterbi    : ['O', 'O', 'O', 'B-LOC', 'O', 'B-LOC']  score=47.7202
  Brute force: ['O', 'O', 'O', 'B-LOC', 'O', 'B-LOC']  score=47.7202
  -> identical: True | Viterbi cost O(T*K^2) instead of O(K^T)

(sklearn-crfsuite reference CRF also trained for comparison)


## TASK 5 - LSTM-based NER model (BiLSTM + char-CNN)

The tagger concatenates a learned word embedding with a character-CNN vector, runs a bidirectional LSTM over the
sentence and predicts a softmax over the BIO tags per token. **The training cell below is the slow step**
(`EPOCHS = 60`, the script default; hyperparameters are not changed).

In [6]:
# ---- TRAINING (BiLSTM + char-CNN). EPOCHS = 60 is the script default (--epochs). ----
EPOCHS = 60
banner("TASK 5 - LSTM-based NER model (BiLSTM + char-CNN)")
lstm = train_lstm(list(train), vocab, epochs=EPOCHS)
n_params = sum(p.numel() for p in lstm.parameters())
print(f"Trainable parameters: {n_params:,}")


TASK 5 - LSTM-based NER model (BiLSTM + char-CNN)


  epoch   1  loss = 13.3497


  epoch  15  loss = 1.4022


  epoch  30  loss = 0.4179


  epoch  45  loss = 0.1252


  epoch  60  loss = 0.2351
BiLSTM trained in 28.2s
Trainable parameters: 94,618


## TASK 6 - Test both models on UNSEEN speech transcripts

The 20 test transcripts mostly contain names, places and organisations that never occur in the training set.
For each one the script prints the out-of-vocabulary (OOV) words and the entities of the gold annotation, the CRF
(Viterbi) prediction and the LSTM prediction. CRF greedy decoding (no transition scores) is also computed for TASK 7.

In [7]:
banner("TASK 6 - Test both models on UNSEEN speech transcripts")
gold = [tags for _, tags in test]
crf_pred = [crf_predict(crf, to_tensor(sent2features(t))) for t, _ in test]
greedy_pred = [crf_predict(crf, to_tensor(sent2features(t)), "greedy") for t, _ in test]
lstm_pred = [lstm_predict(lstm, vocab, t) for t, _ in test]
train_words = {w.lower() for t, _ in train for w in t}
for k, (toks, g) in enumerate(test):
    oov = [w for w in toks if w.lower() not in train_words]
    print(f"\n[{k + 1:02d}] {' '.join(toks)}")
    print(f"     OOV words: {oov if oov else 'none'}")
    print(f"     GOLD : {entities_str(toks, g)}")
    print(f"     CRF  : {entities_str(toks, crf_pred[k])}")
    print(f"     LSTM : {entities_str(toks, lstm_pred[k])}")


TASK 6 - Test both models on UNSEEN speech transcripts



[01] Book a flight for Dr Meera Krishnan from Ahmedabad to Toronto on 12 August
     OOV words: ['Meera', 'Krishnan', 'Ahmedabad', 'Toronto']
     GOLD : Meera Krishnan<PER>, Ahmedabad<LOC>, Toronto<LOC>, 12 August<DATE>
     CRF  : Meera Krishnan<PER>, Ahmedabad<LOC>, Toronto<LOC>, 12 August<DATE>
     LSTM : Meera<PER>, Krishnan<ORG>, Ahmedabad<LOC>, 12 August<DATE>

[02] Schedule a meeting with Rohan Das from Accenture on next Tuesday
     OOV words: ['Rohan', 'Das', 'Accenture']
     GOLD : Rohan Das<PER>, Accenture<ORG>, next Tuesday<DATE>
     CRF  : Rohan Das<PER>, Accenture<LOC>, next Tuesday<DATE>
     LSTM : Rohan Das<PER>, Accenture<ORG>, next Tuesday<DATE>

[03] Call Ananya and tell her I will reach Jaipur tomorrow
     OOV words: ['Ananya', 'her', 'Jaipur']
     GOLD : Ananya<PER>, Jaipur<LOC>, tomorrow<DATE>
     CRF  : Ananya<PER>, Jaipur<LOC>, tomorrow<DATE>
     LSTM : Ananya<PER>, Jaipur<LOC>, tomorrow<DATE>

[04] What is the weather like in Bangkok today
     OOV wo

## TASK 7 - Compare predictions and evaluation metrics

Entity-level micro precision / recall / F1 (exact span and type match) for CRF-Viterbi, CRF-greedy,
sklearn-crfsuite and the BiLSTM, plus the number of invalid BIO transitions (e.g. `O -> I-PER`) each decoder produces.

In [8]:
banner("TASK 7 - Compare predictions and evaluation metrics")
rep_crf = evaluate(gold, crf_pred)
rep_lstm = evaluate(gold, lstm_pred)
rep_greedy = evaluate(gold, greedy_pred)
agree = sum(a == b for a, b in zip(crf_pred, lstm_pred))
print(f"Transcripts where CRF and LSTM agree exactly : {agree}/{len(test)}")
print(f"Tokens where CRF and LSTM agree             : "
      f"{sum(x == y for a, b in zip(crf_pred, lstm_pred) for x, y in zip(a, b))}/"
      f"{sum(len(a) for a in gold)}")
print(f"\n  {'Model':<34}{'Precision':>10}{'Recall':>10}{'F1':>10}{'Tok-Acc':>10}")
rows = [("CRF (Viterbi decoding)", rep_crf), ("CRF (greedy, no transitions)", rep_greedy),
        ("BiLSTM + char-CNN (softmax)", rep_lstm)]
if crfsuite is not None:
    rows.insert(1, ("sklearn-crfsuite (reference)",
                    evaluate(gold, crfsuite.predict([sent2features(t) for t, _ in test]))))
for name, r in rows:
    p, rc, f, _ = r["micro"]
    print(f"  {name:<34}{p:>10.3f}{rc:>10.3f}{f:>10.3f}{r['token_acc']:>10.3f}")

def invalid_bio(seqs):
    return sum(1 for s in seqs for a, b in zip(["O"] + s[:-1], s)
               if b.startswith("I-") and a[2:] != b[2:])
print(f"\nInvalid BIO transitions (e.g. O -> I-PER): CRF-Viterbi={invalid_bio(crf_pred)}, "
      f"CRF-greedy={invalid_bio(greedy_pred)}, LSTM={invalid_bio(lstm_pred)}")


TASK 7 - Compare predictions and evaluation metrics
Transcripts where CRF and LSTM agree exactly : 13/20
Tokens where CRF and LSTM agree             : 186/196

  Model                              Precision    Recall        F1   Tok-Acc
  CRF (Viterbi decoding)                 0.944     0.927     0.936     0.974
  sklearn-crfsuite (reference)           0.945     0.945     0.945     0.985
  CRF (greedy, no transitions)           0.562     0.818     0.667     0.832
  BiLSTM + char-CNN (softmax)            0.875     0.891     0.883     0.964

Invalid BIO transitions (e.g. O -> I-PER): CRF-Viterbi=0, CRF-greedy=11, LSTM=2


## TASK 8 - Analysis of a case where the models disagree

The transcript where the two models differ most in number of errors is selected. For each disagreeing token the
script shows which model is right, whether the word is in the training vocabulary, the CRF feature contributions
(weight x feature value) for its predicted tag, and the LSTM softmax. The interpretation is printed at the end.

In [9]:
banner("TASK 8 - Analysis of a case where the models disagree")
diff_idx = [k for k in range(len(test)) if crf_pred[k] != lstm_pred[k]]
if not diff_idx:
    print("Both models produced identical tag sequences on every test transcript.")
else:
    # prefer a case where exactly one of the models is right
    def n_err(p, g):
        return sum(a != b for a, b in zip(p, g))
    k = max(diff_idx, key=lambda i: abs(n_err(crf_pred[i], gold[i]) - n_err(lstm_pred[i], gold[i])))
    toks = test[k][0]
    print(f"Transcript: {' '.join(toks)}\n")
    print_aligned(toks, [("GOLD", gold[k]), ("CRF", crf_pred[k]), ("LSTM", lstm_pred[k])])
    _, probs = lstm_predict(lstm, vocab, toks, return_probs=True)
    with torch.no_grad():
        x = to_tensor(sent2features(toks)).numpy()
    print()
    for i, tok in enumerate(toks):
        if crf_pred[k][i] == lstm_pred[k][i]:
            continue
        winner = ("CRF" if crf_pred[k][i] == gold[k][i] else
                  "LSTM" if lstm_pred[k][i] == gold[k][i] else "neither")
        print(f"Token '{tok}': gold={gold[k][i]}  CRF={crf_pred[k][i]}  "
              f"LSTM={lstm_pred[k][i]}  -> correct: {winner}")
        print(f"  In training vocabulary: {tok.lower() in train_words}"
              f"{'' if tok.lower() in train_words else '  (LSTM sees <UNK> + char-CNN only)'}")
        ci = TAG2IDX[crf_pred[k][i]]
        contrib = W[ci] * x[i]
        top = np.argsort(-contrib)[:5]
        print(f"  CRF evidence for {crf_pred[k][i]}: "
              + ", ".join(f"{names[j]} ({contrib[j]:+.2f})" for j in top if contrib[j] != 0))
        top_lstm = np.argsort(-probs[i])[:3]
        print("  LSTM softmax: " + ", ".join(f"{IDX2TAG[j]}={probs[i, j]:.2f}" for j in top_lstm))
    print("""
Interpretation:
  * The CRF scores each tag with explicit, human-designed evidence (capitalisation,
    word shape, suffixes, neighbouring words such as 'Dr', 'from', 'in', 'on', month
    names) and Viterbi picks the globally best *sequence*; the learned transition
    scores strongly penalise invalid moves such as O -> I-PER.
  * The BiLSTM must learn everything from 60 sentences. Unseen names are mapped to
    <UNK>, so it relies on the char-CNN and context; its softmax decides each token
    independently, so it can emit inconsistent BIO sequences or confuse ORG/LOC when
    the context ('at', 'in', 'from') is shared by both types.
  * With so little training data, feature engineering + structured decoding (CRF)
    usually generalises better; with thousands of sentences (see Lab 4) the neural
    model catches up and typically overtakes the CRF.""")


TASK 8 - Analysis of a case where the models disagree
Transcript: Book a flight for Dr Meera Krishnan from Ahmedabad to Toronto on 12 August

  Token  Book  a  flight  for  Dr  Meera  Krishnan  from  Ahmedabad  to  Toronto  on  12      August  
  GOLD   O     O  O       O    O   B-PER  I-PER     O     B-LOC      O   B-LOC    O   B-DATE  I-DATE  
  CRF    O     O  O       O    O   B-PER  I-PER     O     B-LOC      O   B-LOC    O   B-DATE  I-DATE  
  LSTM   O     O  O       O    O   B-PER  I-ORG     O     B-LOC      O   O        O   B-DATE  I-DATE  

Token 'Krishnan': gold=I-PER  CRF=I-PER  LSTM=I-ORG  -> correct: CRF
  In training vocabulary: False  (LSTM sees <UNK> + char-CNN only)
  CRF evidence for I-PER: -1:w.istitle (+1.00), 2:w.istitle (+0.68), w.shape=Xxx (+0.52), w.istitle (+0.50), -2:w.is_title_word (+0.49)
  LSTM softmax: I-ORG=0.53, I-PER=0.46, B-PER=0.00
Token 'Toronto': gold=B-LOC  CRF=B-LOC  LSTM=O  -> correct: CRF
  In training vocabulary: False  (LSTM sees <UNK> + char-

## TASK 9 - Evaluation with appropriate metrics

Entity-level exact-match precision / recall / F1 per entity type (CoNLL-2003 style: boundaries and type must both
match), cross-checked with the `seqeval` library. The robustness test lower-cases the test transcripts, as raw ASR
output without true-casing would be, and measures how much each model loses.

In [10]:
banner("TASK 9 - Evaluation with appropriate metrics")
print("Metric: entity-level exact-match Precision / Recall / F1 (CoNLL-2003 style).")
print("An entity counts as correct only if BOTH its boundaries and its type match.")
print_report("CRF (Viterbi) on unseen transcripts", rep_crf)
print_report("BiLSTM + char-CNN on unseen transcripts", rep_lstm)

try:
    from seqeval.metrics import classification_report as seqeval_report
    print("\nCross-check with the seqeval library (CRF):")
    print(seqeval_report(gold, crf_pred, digits=3, zero_division=0))
    print("Cross-check with the seqeval library (LSTM):")
    print(seqeval_report(gold, lstm_pred, digits=3, zero_division=0))
except ImportError:
    print("\n(seqeval not installed - skipping library cross-check)")

# robustness: raw ASR output without true-casing
lower_test = [([w.lower() for w in t], g) for t, g in test]
crf_low = [crf_predict(crf, to_tensor(sent2features(t))) for t, _ in lower_test]
lstm_low = [lstm_predict(lstm, vocab, t) for t, _ in lower_test]
r1, r2 = evaluate(gold, crf_low), evaluate(gold, lstm_low)
print("Robustness test - the same unseen transcripts, lower-cased (ASR without true-casing):")
print(f"  CRF  micro-F1: {rep_crf['micro'][2]:.3f} (cased) -> {r1['micro'][2]:.3f} (lower-cased)")
print(f"  LSTM micro-F1: {rep_lstm['micro'][2]:.3f} (cased) -> {r2['micro'][2]:.3f} (lower-cased)")
drop_crf = rep_crf["micro"][2] - r1["micro"][2]
drop_lstm = rep_lstm["micro"][2] - r2["micro"][2]
more, less = ("CRF", "LSTM") if drop_crf > drop_lstm else ("LSTM", "CRF")
print(f"  The {more} loses more F1: its strongest features/cues are capitalisation-based")
print(f"  (w.istitle, w.shape=Xxx). The {less} is less affected"
      + (" because its word embeddings are built from lower-cased words and it relies more on context."
         if less == "LSTM" else "."))
print("  Real ASR pipelines should true-case the transcript first, or train on lower-cased data.")


TASK 9 - Evaluation with appropriate metrics
Metric: entity-level exact-match Precision / Recall / F1 (CoNLL-2003 style).
An entity counts as correct only if BOTH its boundaries and its type match.

CRF (Viterbi) on unseen transcripts
  Entity     Precision    Recall        F1   Support
  PER            0.917     1.000     0.957        11
  LOC            0.882     0.938     0.909        16
  ORG            1.000     0.750     0.857        12
  DATE           1.000     1.000     1.000        16
  micro avg      0.944     0.927     0.936        55
  macro avg      0.950     0.922     0.931        55
  Token accuracy: 0.974

BiLSTM + char-CNN on unseen transcripts
  Entity     Precision    Recall        F1   Support
  PER            0.750     0.818     0.783        11
  LOC            0.824     0.875     0.848        16
  ORG            0.909     0.833     0.870        12
  DATE           1.000     1.000     1.000        16
  micro avg      0.875     0.891     0.883        55
  macro av

Robustness test - the same unseen transcripts, lower-cased (ASR without true-casing):
  CRF  micro-F1: 0.936 (cased) -> 0.451 (lower-cased)
  LSTM micro-F1: 0.883 (cased) -> 0.833 (lower-cased)
  The CRF loses more F1: its strongest features/cues are capitalisation-based
  (w.istitle, w.shape=Xxx). The LSTM is less affected because its word embeddings are built from lower-cased words and it relies more on context.
  Real ASR pipelines should true-case the transcript first, or train on lower-cased data.


## Final demo - the README `--text` example

Runs the README example `call Ravi Kumar from Infosys in Pune on 3 March` (not in the training set) through both
trained models, using the same pipeline as `python ner_crf_lstm.py --text "..."`, and shows the tags as a table.

In [11]:
from IPython.display import Markdown, display

TEXT = "call Ravi Kumar from Infosys in Pune on 3 March"
toks = tokenize(strip_markup(TEXT))
banner(f"Custom transcript: {' '.join(toks)}")
c = crf_predict(crf, to_tensor(sent2features(toks)))
l = lstm_predict(lstm, vocab, toks)
print_aligned(toks, [("CRF", c), ("LSTM", l)])
print(f"\n  CRF entities : {entities_str(toks, c)}")
print(f"  LSTM entities: {entities_str(toks, l)}")

table = ["| Token | CRF tag | LSTM tag |", "|---|---|---|"]
table += [f"| {w} | {a} | {b} |" for w, a, b in zip(toks, c, l)]
display(Markdown("\n".join(table)))


Custom transcript: call Ravi Kumar from Infosys in Pune on 3 March
  Token  call  Ravi   Kumar  from  Infosys  in  Pune   on  3       March   
  CRF    O     B-PER  I-PER  O     B-ORG    O   B-LOC  O   B-DATE  I-DATE  
  LSTM   O     B-PER  I-PER  O     B-ORG    O   B-LOC  O   B-DATE  I-DATE  

  CRF entities : Ravi Kumar<PER>, Infosys<ORG>, Pune<LOC>, 3 March<DATE>
  LSTM entities: Ravi Kumar<PER>, Infosys<ORG>, Pune<LOC>, 3 March<DATE>


| Token | CRF tag | LSTM tag |
|---|---|---|
| call | O | O |
| Ravi | B-PER | B-PER |
| Kumar | I-PER | I-PER |
| from | O | O |
| Infosys | B-ORG | B-ORG |
| in | O | O |
| Pune | B-LOC | B-LOC |
| on | O | O |
| 3 | B-DATE | B-DATE |
| March | I-DATE | I-DATE |

### Optional: `--audio` (needs internet)

`python ner_crf_lstm.py --audio my_command.wav` transcribes a WAV/FLAC file with the free Google Speech
Recognition web API (via the `SpeechRecognition` package) and then tags the transcript. This **requires an internet
connection**, so it is disabled by default: set `AUDIO_FILE` to a WAV path to run it. Errors are caught so that the
notebook still runs to the end without internet.

In [12]:
# Optional: transcribe a WAV/FLAC with Google STT (needs internet), then tag it.
# Equivalent of: python ner_crf_lstm.py --audio my_command.wav
AUDIO_FILE = None          # e.g. ROOT / "my_command.wav"
try:
    if AUDIO_FILE is not None:
        txt = transcribe_audio(str(AUDIO_FILE))
        if txt:
            print(f"Transcribed audio: {txt}")
            toks = tokenize(strip_markup(txt))
            banner(f"Custom transcript: {' '.join(toks)}")
            c = crf_predict(crf, to_tensor(sent2features(toks)))
            l = lstm_predict(lstm, vocab, toks)
            print_aligned(toks, [("CRF", c), ("LSTM", l)])
            print(f"\n  CRF entities : {entities_str(toks, c)}")
            print(f"  LSTM entities: {entities_str(toks, l)}")
    else:
        print("Audio step skipped (AUDIO_FILE is None).")
except Exception as e:  # missing package, no internet, bad file
    print(f"Audio step skipped: {e}")

Audio step skipped (AUDIO_FILE is None).


## Notes / what to look for

- **TASK 1-2:** 60 training and 20 unseen test transcripts are converted to BIO tags (9 tags). The CRF uses 2,022 sparse features; the LSTM vocabulary has 298 words and 61 characters.
- **TASK 3-4:** The from-scratch CRF reaches micro-F1 0.936 on the unseen transcripts. Viterbi matches brute-force enumeration on the 6-token check sentence (9^6 = 531,441 paths). The sklearn-crfsuite reference gets 0.945.
- **TASK 5:** The BiLSTM + char-CNN (94,618 parameters) reaches micro-F1 0.883. Its printed epoch loss is not monotone (epoch 45: 0.125, epoch 60: 0.235); dropout and random word dropout are active during training, so the printed loss is noisy.
- **TASK 6-7:** CRF-Viterbi and the LSTM agree exactly on 13/20 transcripts (186/196 tokens). CRF-Viterbi produces 0 invalid BIO transitions, the LSTM 2, and CRF-greedy 11. Greedy CRF decoding (no transition scores) drops to F1 0.667, which shows the value of the transition matrix.
- **TASK 8:** On `Dr Meera Krishnan`, the LSTM tags `Krishnan` as I-ORG (gold I-PER) and misses `Toronto`; the CRF is correct on both. Both words are out of vocabulary for the LSTM.
- **TASK 9:** The seqeval cross-check reproduces the same micro-F1 values (0.936 CRF, 0.883 LSTM). Lower-casing the test transcripts drops the CRF to F1 0.451 and the LSTM to 0.833, so the CRF depends much more on capitalisation.
- **Demo:** For `call Ravi Kumar from Infosys in Pune on 3 March`, both models tag Ravi Kumar (PER), Infosys (ORG), Pune (LOC) and 3 March (DATE) identically.
- The README quotes CRF F1 approx. 0.94 and BiLSTM approx. 0.88; the executed run gives 0.936 and 0.883. The run is seeded, but exact LSTM numbers can shift slightly with the CPU/thread setup.